# Double Texting

无缝处理[double texting](https://langchain-ai.github.io/langgraph/concepts/double_texting/)对于处理实际使用场景很重要，特别是在聊天应用中。

用户可以在先前的 run(s) 完成之前连续发送多条消息，我们希望确保优雅地处理这种情况。

## 拒绝

一个简单的方法是[拒绝](https://langchain-ai.github.io/langgraph/cloud/how-tos/reject_concurrent/)任何新的 runs，直到当前 run 完成。

In [ ]:
%%capture --no-stderr
%pip install -U langgraph_sdk

In [ ]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

In [ ]:
import httpx
from langchain_core.messages import HumanMessage

# 创建一个线程
thread = await client.threads.create()

# 创建待办事项
user_input_1 = "添加一个待办事项来跟进DI Repairs。"
user_input_2 = "添加一个待办事项将梳妆台固定到墙上。"
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)
try:
    await client.runs.create(
        thread["thread_id"],
        graph_name,
        input={"messages": [HumanMessage(content=user_input_2)]}, 
        config=config,
        multitask_strategy="reject",
    )
except httpx.HTTPStatusError as e:
    print("启动并发运行失败", e)

In [ ]:
from langchain_core.messages import convert_to_messages

# 等待原始 run 完成
await client.runs.join(thread["thread_id"], run["run_id"])

# 获取线程的状态
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

## 排队

我们可以使用[排队](https://langchain-ai.github.io/langgraph/cloud/how-tos/enqueue_concurrent/)任何新的 runs，直到当前 run 完成。

In [ ]:
# 创建新线程
thread = await client.threads.create()

# 创建新的待办事项
user_input_1 = "这个周末给Erik寄他的T恤礼物。"
user_input_2 = "取现金并支付保姆2周的工资。在周五前完成。"
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

first_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="enqueue",
)

# 等待第二个 run 完成
await client.runs.join(thread["thread_id"], second_run["run_id"])

# 获取线程的状态
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

## 中断

我们可以使用[中断](https://langchain-ai.github.io/langgraph/cloud/how-tos/interrupt_concurrent/)来中断当前 run，但保存到该点为止完成的所有工作。

In [ ]:
import asyncio

# 创建新线程
thread = await client.threads.create()

# 创建新的待办事项
user_input_1 = "给我明天到期的待办事项摘要。"
user_input_2 = "算了，创建一个待办事项，在下周五前订购感恩节火腿。"
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

interrupted_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

# 等待 run 1 的一部分完成，这样我们可以在线程中看到它
await asyncio.sleep(1)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="interrupt",
)

# 等待第二个 run 完成
await client.runs.join(thread["thread_id"], second_run["run_id"])

# 获取线程的状态
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

我们可以看到初始 run 被保存，状态为 `interrupted`。

In [ ]:
# 确认第一个 run 被中断
print((await client.runs.get(thread["thread_id"], interrupted_run["run_id"]))["status"])

## 回滚

我们可以使用[回滚](https://langchain-ai.github.io/langgraph/cloud/how-tos/rollback_concurrent/)来中断图的先前 run，删除它，并使用 double-texted 输入启动新 run。

In [ ]:
# 创建新线程
thread = await client.threads.create()

# 创建新的待办事项
user_input_1 = "添加一个待办事项打电话预约瑜伽。"
user_input_2 = "实际上，添加一个待办事项在周日亲自去瑜伽馆。"
config = {"configurable": {"user_id": "Test-Double-Texting"}}
graph_name = "task_maistro" 

rolled_back_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_1)]}, 
    config=config,
)

second_run = await client.runs.create(
    thread["thread_id"],
    graph_name,
    input={"messages": [HumanMessage(content=user_input_2)]}, 
    config=config,
    multitask_strategy="rollback",
)

# 等待第二个 run 完成
await client.runs.join(thread["thread_id"], second_run["run_id"])

# 获取线程的状态
state = await client.threads.get_state(thread["thread_id"])
for m in convert_to_messages(state["values"]["messages"]):
    m.pretty_print()

初始 run 被删除。

In [ ]:
# 确认原始 run 被删除
try:
    await client.runs.get(thread["thread_id"], rolled_back_run["run_id"])
except httpx.HTTPStatusError as _:
    print("原始 run 被正确删除")

### 总结

我们可以看到[所有方法的总结](https://langchain-ai.github.io/langgraph/concepts/double_texting/)：

![Screenshot 2024-11-15 at 12.13.18 PM.png](attachment:ff0af98b-71b1-497a-9c0e-b3519662fd2c.png)